In [3]:
import os
import torch

from datasets import load_dataset

from transformers.utils.import_utils import is_flash_attn_2_available

from colpali_engine.models import ColQwen2, ColQwen2Processor
#from colpali_engine.models import ColPali, ColPaliProcessor # uncomment if you prefer to use ColPali models instead of ColQwen2 models

import weaviate




In [4]:
# colpali inference wrapper

# Get rid of process forking deadlock warnings.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available(): # If GPU available
    device = "cuda:0"
elif torch.backends.mps.is_available(): # If Apple Silicon available
    device = "mps"
else:
    device = "cpu"

if is_flash_attn_2_available():
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "eager"

print(f"Using device: {device}")
print(f"Using attention implementation: {attn_implementation}")

model_name = "vidore/colqwen2-v1.0"

# About a 5 GB download and similar memory usage.
model = ColQwen2.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation=attn_implementation,
).eval()

# Load processor
processor = ColQwen2Processor.from_pretrained(model_name)

# A convenience class to wrap the embedding functionality 
# of ColVision models like ColPali and ColQwen2 
class ColVision:
    def __init__(self, model, processor):
        """Initialize with a loaded model and processor."""
        self.model = model
        self.processor = processor

    # A batch size of one appears to be most performant when running on an M4.
    # Note: Reducing the image resolution speeds up the vectorizer and produces
    # fewer multi-vectors.
    def multi_vectorize_image(self, img):
        """Return the multi-vector image of the supplied PIL image."""
        image_batch = self.processor.process_images([img]).to(self.model.device)
        with torch.no_grad():
            image_embedding = self.model(**image_batch)
        return image_embedding[0]

    def multi_vectorize_text(self, query):
        """Return the multi-vector embedding of the query text string."""
        query_batch = self.processor.process_queries([query]).to(self.model.device)
        with torch.no_grad():
            query_embedding = self.model(**query_batch)
        return query_embedding[0]

# Instantiate the model to be used below.
colvision_embedder = ColVision(model, processor) # This will be instantiated after loading the model and processor

`torch_dtype` is deprecated! Use `dtype` instead!


Using device: mps
Using attention implementation: eager


Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.16s/it]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [5]:
queries = [
    "A table with LLM benchmark results.",
    "A figure detailing the architecture of a neural network.",
]

query_embeddings = [colvision_embedder.multi_vectorize_text(q) for q in queries]
print(query_embeddings[0].shape)  # torch.Size([20, 128])

torch.Size([18, 128])


In [7]:
# connect to collection
# Weaviate Cloud Deployment
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY")),
)

print(weaviate_client.is_ready())

True


In [8]:
colvision_collection = weaviate_client.collections.get("IRPapers_ColQwen2")

def colvision_query(query: str) -> list[str]:    
    near_vector = colvision_embedder.multi_vectorize_text(query).cpu().float().numpy()

    response = colvision_collection.query.near_vector(
        near_vector=near_vector,
        target_vector="colqwen",
        limit=20,
    )
    
    results = []
    for i, o in enumerate(response.objects):
        p = o.properties
        results.append(p["dataset_id"])

    return results

In [11]:
# load dataset
queries = load_dataset("weaviate/irpapers-queries")["train"]

queries[0]

{'dataset_id': '1_1',
 'pdf_id': 1,
 'pdf_title': 'Can Generative LLMs Create Query Variants for Test Collections?',
 'page_number': 1,
 'question': 'What percentage overlap in document pooling at depth 100 did GPT-3.5 generated query variants achieve compared to human-generated variants in the UQV100 test collection?',
 'answer': 'GPT-3.5 generated query variants achieved up to 71.1% overlap with human-generated variants at a pool depth of 100.'}

In [13]:
results = colvision_query(queries[0]["question"])
print(results)

['1_2', '1_3', '1_4', '95_12', '1_1', '28_21', '95_13', '60_4', '45_8', '115_4', '18_4', '99_24', '28_17', '109_14', '91_13', '60_2', '114_9', '158_3', '53_7', '93_17']


In [15]:
# load metric
def compute_success_at_k(results: list[str], gold_id: str, k: int) -> bool:
    return gold_id in results[:k]

compute_success_at_k(results, queries[0]["dataset_id"], 5)

True

In [16]:
# run eval
import time

successes_at_1, successes_at_5, successes_at_20 = 0, 0, 0
query_times = []

experiment_start = time.time()
for idx, query in enumerate(queries):
    if idx % 20 == 19:
        print(f"Processing query {idx + 1} of {len(queries)}")
        print(f"Success rate at 1: {successes_at_1 / (idx + 1)}")
        print(f"Success rate at 5: {successes_at_5 / (idx + 1)}")
        print(f"Success rate at 20: {successes_at_20 / (idx + 1)}")
        print(f"Experiment running for {time.time() - experiment_start:.2f} seconds")
        print(f"Average query time: {sum(query_times) / len(query_times):.2f} seconds")

    start_time = time.time()
    results = colvision_query(query["question"])
    query_time = time.time() - start_time
    query_times.append(query_time)

    successes_at_1 += compute_success_at_k(results, query["dataset_id"], 1)
    successes_at_5 += compute_success_at_k(results, query["dataset_id"], 5)
    successes_at_20 += compute_success_at_k(results, query["dataset_id"], 20)

print(f"Success rate at 1: {successes_at_1 / len(queries)}")
print(f"Success rate at 5: {successes_at_5 / len(queries)}")
print(f"Success rate at 20: {successes_at_20 / len(queries)}")

Processing query 20 of 180
Success rate at 1: 0.55
Success rate at 5: 0.8
Success rate at 20: 0.8
Experiment running for 32.53 seconds
Average query time: 1.71 seconds
Processing query 40 of 180
Success rate at 1: 0.4
Success rate at 5: 0.75
Success rate at 20: 0.9
Experiment running for 64.77 seconds
Average query time: 1.66 seconds
Processing query 60 of 180
Success rate at 1: 0.48333333333333334
Success rate at 5: 0.7666666666666667
Success rate at 20: 0.9166666666666666
Experiment running for 96.15 seconds
Average query time: 1.63 seconds
Processing query 80 of 180
Success rate at 1: 0.4875
Success rate at 5: 0.7875
Success rate at 20: 0.9125
Experiment running for 131.52 seconds
Average query time: 1.66 seconds
Processing query 100 of 180
Success rate at 1: 0.49
Success rate at 5: 0.78
Success rate at 20: 0.93
Experiment running for 158.05 seconds
Average query time: 1.60 seconds
Processing query 120 of 180
Success rate at 1: 0.49166666666666664
Success rate at 5: 0.775
Success ra